# 05 · NLI check (RoBERTa-MNLI and the 0.5 threshold)

Step 1 of the run plan (owner: Cassey). Before the 100-debate run, confirm that the real
`roberta-large-mnli` scorer behaves sensibly and that `nli.contradiction_threshold: 0.5` in
`configs/config.yaml` is a reasonable cut.

The same threshold is used in three places, all through `symmetric_contradiction_batch`
(both NLI directions, larger score wins):

* **C2 stance filter**: drops a passage that contradicts the bot's claim log.
* **D reranker**: picks the candidate whose worst contradiction is lowest.
* **Contradiction rate**: the main metric (C1 against B).

This notebook calls that same function, so what you see here is what the pipeline does.
It changes no code or config. Runs on Colab (GPU: *Runtime → Change runtime type → T4*);
CPU works too, just slower.

## Setup

Finds the repo root, puts `src/` on the path, installs missing deps (same cell as notebook 03).

In [1]:
import importlib.util, os, subprocess, sys
from pathlib import Path

# Colab check that does NOT raise when the `google` package is absent (it does on the
# SCIS cluster / a local venv). find_spec on a dotted name raises if the parent is missing.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# On Colab, clone the repo once. Elsewhere (cluster / local) it is already present.
if IN_COLAB:
    REPO_URL = "https://github.com/diyadj/argubot.git"
    if not os.path.isdir("argubot"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("argubot")

# Work from the repo root so configs/config.yaml and data/... resolve. Walk up from the launch
# dir (notebooks/ on the cluster), and fall back to ~/argubot (execnotebook runs from your home).
root = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "pyproject.toml").exists()), None)
if root is None and (Path.home() / "argubot" / "pyproject.toml").exists():
    root = Path.home() / "argubot"
if root is not None:
    os.chdir(root)

src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)

pip_name = {"torch": "torch", "transformers": "transformers", "numpy": "numpy", "yaml": "pyyaml"}
need = [pip_name[m] for m in pip_name if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

for _m in [k for k in list(sys.modules) if k == "argubot" or k.startswith("argubot.")]:
    del sys.modules[_m]

import argubot  # noqa: F401
print("cwd:", os.getcwd(), "| IN_COLAB:", IN_COLAB, "| argubot:", argubot.__file__)

cwd: /content/argubot | IN_COLAB: True | argubot: /content/argubot/src/argubot/__init__.py


## Load the real scorer

Model name and `max_length` come from the config, as in `scripts/evaluate.py`.

In [2]:
import time
from argubot.config import load_config
from argubot.nli.scorer import RobertaMNLIScorer, symmetric_contradiction_batch

cfg = load_config("configs/config.yaml")
THRESHOLD = cfg["nli"]["contradiction_threshold"]

t0 = time.time()
scorer = RobertaMNLIScorer(cfg["nli"]["model"], max_length=cfg["nli"]["max_length"])
print(f"loaded {cfg['nli']['model']} on {scorer.device} in {time.time() - t0:.0f}s")
print("label order from model config:", scorer.model.config.id2label)
print("threshold in config:", THRESHOLD)

# Quick sanity check: an obvious contradiction should score high, an obvious agreement low.
s1 = scorer.score("School uniforms reduce bullying.", "School uniforms do not reduce bullying.")
s2 = scorer.score("School uniforms reduce bullying.", "Uniforms cut down on bullying in schools.")
print("obvious contradiction:", s1)
print("obvious agreement    :", s2)

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded roberta-large-mnli on cuda in 38s
label order from model config: {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}
threshold in config: 0.5
obvious contradiction: NLIScore(contradiction=0.9991148114204407, neutral=0.0006379749393090606, entailment=0.00024726532865315676)
obvious agreement    : NLIScore(contradiction=0.0013852345291525126, neutral=0.0049153282307088375, entailment=0.9936994314193726)


## 1. Score 24 hand-labelled claim/passage pairs

Each pair is *(earlier bot claim, new passage or sentence)*. `should_flag` is what a person
would say: `True` if the second text contradicts the claim, `False` if it agrees, is unrelated,
or argues a **different point** for the other side. The last kind matters most for C2: a passage
for the other side that doesn't contradict anything the bot said should be kept.

Add or change pairs freely. A few real ones from the 3-debate check (27 Oct) would make this stronger.

In [3]:
# (claim, text, should_flag)
PAIRS = [
    # School uniforms: the toy passages in data/toy/toy_passages.jsonl
    ("School uniforms reduce bullying in schools.",
     "School uniforms reduce bullying in schools because clothing differences matter less.", False),
    ("School uniforms reduce bullying in schools.",
     "School uniforms do not reduce bullying in schools, and some studies find no effect.", True),
    ("School uniforms reduce bullying in schools.",
     "Uniforms save families money every year because they replace fashionable clothes.", False),
    ("Uniforms help students express themselves.",
     "Uniforms limit self expression, which matters for the development of students.", True),
    ("Uniforms save families money.",
     "Uniforms limit self expression, which matters for the development of students.", False),
    ("There is strong evidence that uniforms raise grades.",
     "Research linking uniforms to higher grades is weak and often confounded by other school policies.", True),
    # Death penalty
    ("The death penalty deters crime.",
     "States with capital punishment do not have lower murder rates than states without it.", True),
    ("The death penalty deters crime.",
     "Executing murderers discourages others from committing murder.", False),
    ("The death penalty is cheaper than life imprisonment.",
     "Death penalty cases cost far more than life sentences because of the long appeals.", True),
    ("Innocent people have been executed.",
     "The death penalty is expensive to administer.", False),
    # Minimum wage
    ("Raising the minimum wage costs jobs.",
     "Higher minimum wages have not led to job losses.", True),
    ("Raising the minimum wage costs jobs.",
     "Small businesses often cut staff hours when wages go up.", False),
    # Nuclear power
    ("Nuclear power is safe.",
     "Nuclear energy has one of the lowest death rates per unit of electricity.", False),
    ("Nuclear power is safe.",
     "Nuclear plants are dangerous, as Chernobyl and Fukushima showed.", True),
    ("Nuclear power is too expensive.",
     "Nuclear power is safe.", False),
    # Mixed topics
    ("Homework improves learning in primary school.",
     "For young children, homework has almost no effect on achievement.", True),
    ("Video games make children violent.",
     "Video games are a popular hobby around the world.", False),
    ("Social media harms teenagers' mental health.",
     "Social media helps teenagers stay connected with their friends.", False),
    ("Animal testing is necessary for medical progress.",
     "Most medical advances could not have happened without animal research.", False),
    ("Animal testing is necessary for medical progress.",
     "We no longer need animal testing, since modern alternatives work just as well.", True),
    ("Zoos protect endangered species.",
     "Zoos play no meaningful role in conservation.", True),
    ("I support a four-day work week.",
     "I am against shortening the work week.", True),
    ("Remote work increases productivity.",
     "Some workers feel lonely when working from home.", False),
    ("Voting should be compulsory.",
     "Nobody should be forced to vote.", True),
]
print(len(PAIRS), "pairs,", sum(f for *_, f in PAIRS), "should be flagged")

24 pairs, 12 should be flagged


In [4]:
pairs = [(c, t) for c, t, _ in PAIRS]
expected = [f for *_, f in PAIRS]

# One batch: forward (claim -> text) and backward (text -> claim), so both directions are visible.
both = scorer.score_batch(pairs + [(t, c) for c, t in pairs])
fwd = [s.contradiction for s in both[: len(pairs)]]
bwd = [s.contradiction for s in both[len(pairs):]]
sym = symmetric_contradiction_batch(scorer, pairs)   # what the pipeline uses
assert all(abs(s - max(f, b)) < 1e-4 for s, f, b in zip(sym, fwd, bwd))

def short(s, n=55):
    return s if len(s) <= n else s[: n - 1] + "…"

print(f"{'#':>2}  {'claim':<40} {'text':<55} {'fwd':>5} {'bwd':>5} {'max':>5}  exp  got")
for i, ((c, t), f, b, s, e) in enumerate(zip(pairs, fwd, bwd, sym, expected), 1):
    got = s >= THRESHOLD
    mark = "" if got == e else "  <-- WRONG"
    print(f"{i:>2}  {short(c, 40):<40} {short(t):<55} {f:5.2f} {b:5.2f} {s:5.2f}  {'Y' if e else 'n':>3}  {'Y' if got else 'n':>3}{mark}")

wrong = sum((s >= THRESHOLD) != e for s, e in zip(sym, expected))
print(f"\nat threshold {THRESHOLD}: {len(PAIRS) - wrong}/{len(PAIRS)} correct")

 #  claim                                    text                                                      fwd   bwd   max  exp  got
 1  School uniforms reduce bullying in scho… School uniforms reduce bullying in schools because clo…  0.00  0.00  0.00    n    n
 2  School uniforms reduce bullying in scho… School uniforms do not reduce bullying in schools, and…  0.99  1.00  1.00    Y    Y
 3  School uniforms reduce bullying in scho… Uniforms save families money every year because they r…  0.01  0.03  0.03    n    n
 4  Uniforms help students express themselv… Uniforms limit self expression, which matters for the …  1.00  1.00  1.00    Y    Y
 5  Uniforms save families money.            Uniforms limit self expression, which matters for the …  0.45  0.32  0.45    n    n
 6  There is strong evidence that uniforms … Research linking uniforms to higher grades is weak and…  0.98  1.00  1.00    Y    Y
 7  The death penalty deters crime.          States with capital punishment do not have lower mur

## 2. Is 0.5 a good threshold?

For each threshold: **precision** = of the pairs flagged, how many really contradict;
**recall** = of the real contradictions, how many got flagged. If a wide range around 0.5 gives
the same result, the exact value doesn't matter much and 0.5 can stay.

In [5]:
print(f"{'thr':>5} {'flagged':>8} {'precision':>10} {'recall':>8} {'accuracy':>9}")
for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    got = [s >= thr for s in sym]
    tp = sum(g and e for g, e in zip(got, expected))
    fp = sum(g and not e for g, e in zip(got, expected))
    fn = sum(e and not g for g, e in zip(got, expected))
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    acc = sum(g == e for g, e in zip(got, expected)) / len(expected)
    mark = "   <- config" if abs(thr - THRESHOLD) < 1e-9 else ""
    print(f"{thr:5.1f} {sum(got):8d} {prec:10.2f} {rec:8.2f} {acc:9.2f}{mark}")

pos = sorted(s for s, e in zip(sym, expected) if e)
neg = sorted(s for s, e in zip(sym, expected) if not e)
print(f"\nreal contradictions:  min {pos[0]:.2f}, median {pos[len(pos) // 2]:.2f}")
print(f"non-contradictions:   max {neg[-1]:.2f}, median {neg[len(neg) // 2]:.2f}")
print("(a clean gap between the two means the threshold is safe anywhere inside it)")

  thr  flagged  precision   recall  accuracy
  0.1       17       0.71     1.00      0.79
  0.2       16       0.75     1.00      0.83
  0.3       16       0.75     1.00      0.83
  0.4       16       0.75     1.00      0.83
  0.5       15       0.80     1.00      0.88   <- config
  0.6       15       0.80     1.00      0.88
  0.7       15       0.80     1.00      0.88
  0.8       13       0.92     1.00      0.96
  0.9       12       0.92     0.92      0.92

real contradictions:  min 0.87, median 1.00
non-contradictions:   max 0.93, median 0.08
(a clean gap between the two means the threshold is safe anywhere inside it)


## 3. Sanity check on real debate text

Runs the actual metric, `contradiction_rate`, on the human **bot side** of the first 20 scripted
test debates (`data/processed/test_dialogues.jsonl`). People in CMV rarely contradict themselves
outright, so a very high rate here means the scorer or threshold is too eager. Then read the
flagged pairs: are they real contradictions, or NLI mistakes?

Uses the regex claim extractor if spaCy isn't installed. That's fine for this check.

In [6]:
from argubot.eval.metrics import contradiction_rate
from argubot.io import read_jsonl
from argubot.schemas import Transcript, TurnRecord

N_DIALOGUES = 20

transcripts = []
for d in list(read_jsonl("data/processed/test_dialogues.jsonl"))[:N_DIALOGUES]:
    turns, user = [], ""
    for t in d["turns"]:
        if t["speaker"] == "user":
            user = t["text"]
        else:
            turns.append(TurnRecord(user_text=user, bot_text=t["text"], claims_in_prompt=[],
                                    passages_in_prompt=[], candidates=[], prompt=""))
    transcripts.append(Transcript(dialogue_id=d["dialogue_id"], system="human", turns=turns))

t0 = time.time()
res = contradiction_rate(transcripts, scorer, threshold=THRESHOLD)
print(f"human CMV bot side: contradiction rate {res.rate:.3f} "
      f"({res.n_flagged}/{res.n_turns} turns flagged, {time.time() - t0:.0f}s)")

human CMV bot side: contradiction rate 0.706 (24/34 turns flagged, 25s)


In [7]:
# Highest-scoring flagged pairs first. Mark each one: real contradiction, or NLI error?
for f in sorted(res.flagged, key=lambda x: -x["score"])[:15]:
    print(f"[{f['score']:.2f}] {f['dialogue_id']} turn {f['turn']}")
    print("   earlier:", short(f["earlier_claim"], 150))
    print("   new    :", short(f["new_sentence"], 150))
    print()

[1.00] t3_3igdu9_2 turn 1
   earlier: Basic rights exist without intervention from anyone else.
   new    : Rights do not.

[1.00] t3_3igdu9_2 turn 2
   earlier: Basic rights exist without intervention from anyone else.
   new    : Society has to grant you that right.

[1.00] t3_3igdu9_2 turn 2
   earlier: They are completely different things.
   new    : Indeed it was exactly what you said.

[1.00] t3_3igdu9_2 turn 2
   earlier: You are confusing natural rights with granted rights.
   new    : Indeed it was exactly what you said.

[1.00] t3_3igdu9_2 turn 1
   earlier: Basic rights exist without intervention from anyone else.
   new    : A social construct requires agreement from other people.

[1.00] t3_3igdu9_2 turn 2
   earlier: They are completely different things.
   new    : It's pretty simple - can you do it without harming or requiring another person?

[1.00] t3_3igdu9_2 turn 1
   earlier: Those people still have that right.
   new    : Rights do not.

[1.00] t3_2yo9nm_5 turn 2

## What to record

Fill in after running, then paste into the results section of the report and tell the group
whether the threshold stays.

* Device and load time:
* Section 1, pairs correct at 0.5: __ / 24. Which ones were wrong, and why:
* Section 2, range of thresholds with the same result: __ to __. **Keep 0.5?** yes / no
* Section 3, human contradiction rate: __. Of the top flagged pairs, real contradictions: __ / __
* Anything odd (e.g. one direction much higher than the other):